<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/Sz%C3%B6veg%20Klasszifik%C3%A1ci%C3%B3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Szöveg Klasszifikáció

**Licenc: CC BY-NC-SA 4.0**

A **szövegosztályozás** (text classification) az NLP egyik alapfeladata: szöveges dokumentumok besorolása előre definiált kategóriákba.

## Gyakorlati alkalmazások

| Feladat | Bemenet | Kimenet |
|---------|---------|---------|
| **Spam detection** | Email | spam / nem spam |
| **Sentiment analysis** | Vélemény | pozitív / negatív / semleges |
| **Topic classification** | Cikk | sport / politika / tech |
| **Intent detection** | Chatbot query | üdvözlés / kérdés / panasz |
| **Toxicity detection** | Komment | toxikus / normál |

## Tartalomjegyzék
1. Tokenizálás és előfeldolgozás
2. Bag of Words / TF-IDF (klasszikus)
3. Neural szövegosztályozás (CNN, RNN)
4. Transformer-alapú osztályozás
5. Teljes pipeline gyakorlat

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

torch.manual_seed(42)

## 1. Tokenizálás és előfeldolgozás

A szöveget feldolgozható formátumra kell alakítani. Ez a **tokenizálás**: szöveg → token lista.

### Tokenizálási stratégiák

| Módszer | Példa: "futottam" | Előny | Hátrány |
|---------|-------------------|-------|---------|
| **Szó-alapú** | ["futottam"] | Egyszerű | Nagy szókincs, OOV |
| **Karakter-alapú** | ["f","u","t",...] | Nincs OOV | Hosszú, kevés szemantika |
| **Subword (BPE)** | ["fut", "##ottam"] | Kompromisszum | Komplexebb |

### Előfeldolgozás lépések

1. **Lowercase**: "Kutya" → "kutya"
2. **Punctuation removal**: "hello!" → "hello"
3. **Stopword removal**: "a kutya fut" → "kutya fut"
4. **Stemming/Lemmatization**: "futottam" → "fut"

In [ ]:
texts = [
    "Ez egy pozitív vélemény a termékről",
    "Nagyon rossz minőség sajnos",
    "Kiváló termék ajánlom mindenkinek",
    "Nem működik csalódás"
]
labels = [1, 0, 1, 0]  # 1=pozitív, 0=negatív

# Egyszerű tokenizálás
def tokenize(text):
    return text.lower().split()

# Szótár építése
all_tokens = [tok for text in texts for tok in tokenize(text)]
vocab = {word: i+1 for i, word in enumerate(set(all_tokens))}  # 0 = padding
vocab['<PAD>'] = 0

print(f"Vocab size: {len(vocab)}")
print(f"Sample: {list(vocab.items())[:5]}")

## 2. Bag of Words / TF-IDF (klasszikus módszerek)

### Bag of Words (BoW)

A dokumentum reprezentációja: egy vektor, ahol minden dimenzió egy szó előfordulási száma.

**Probléma**: Gyakori szavak ("a", "és") nagy súlyt kapnak, de kevés információt hordoznak.

### TF-IDF (Term Frequency - Inverse Document Frequency)

A ritka, de releváns szavakat súlyozza fel:

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \log\frac{N}{\text{DF}(t)}$$

Ahol:
- $\text{TF}(t, d)$: szó gyakorisága a dokumentumban
- $\text{DF}(t)$: hány dokumentumban fordul elő
- $N$: összes dokumentum száma

**Intuíció**: "kutya" ami 1 dokumentumban van 10-ből → nagy súly. "a" ami mind a 10-ben → kis súly.

In [ ]:
def bag_of_words(text, vocab):
    bow = np.zeros(len(vocab))
    for token in tokenize(text):
        if token in vocab:
            bow[vocab[token]] += 1
    return bow

# BoW reprezentáció
X_bow = np.array([bag_of_words(t, vocab) for t in texts])
print(f"BoW shape: {X_bow.shape}")

# Vizualizáció
plt.figure(figsize=(10, 3))
plt.imshow(X_bow, aspect='auto', cmap='Blues')
plt.xlabel('Vocab index'); plt.ylabel('Document')
plt.title('Bag of Words')
plt.colorbar(label='Count')
plt.show()

## 3. Neural szövegosztályozás

A neurális megközelítések tanult word embeddings-eket használnak, és megtartják a szavak sorrendjét.

### TextCNN (Kim, 2014)

**Ötlet**: 1D konvolúciós filterek különböző méretekkel (n-gram minták detektálása).

```
Input → Embedding → Conv1D(3,4,5) → MaxPool → Concat → FC → Output
         [B,L,D]    Multiple filters   Global
```

**Miért működik?**
- Filter size 3: trigram minták ("nagyon jó termék")
- Filter size 4: 4-gram minták
- Max pooling: a legerősebb minta kiválasztása

### BiLSTM

**Ötlet**: Kétirányú LSTM a teljes kontextus megragadására.

- **Forward LSTM**: balról jobbra olvas
- **Backward LSTM**: jobbról balra olvas
- **Concat**: mindkét irány összefűzése

In [ ]:
class TextCNN(nn.Module):
    """1D CNN szövegosztályozáshoz."""
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.conv1 = nn.Conv1d(embed_dim, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(embed_dim, 32, kernel_size=4, padding=1)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        # x: [B, seq_len]
        emb = self.embedding(x).transpose(1, 2)  # [B, D, L]

        c1 = F.relu(self.conv1(emb)).max(dim=2)[0]  # Global max pool
        c2 = F.relu(self.conv2(emb)).max(dim=2)[0]

        concat = torch.cat([c1, c2], dim=1)
        return self.fc(concat)

model = TextCNN(vocab_size=100, embed_dim=32, num_classes=2)
x = torch.randint(0, 100, (4, 20))
print(f"Input: {x.shape} → Output: {model(x).shape}")

In [ ]:
class TextRNN(nn.Module):
    """LSTM szövegosztályozáshoz."""
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        emb = self.embedding(x)  # [B, L, D]
        _, (h, _) = self.lstm(emb)
        # Concat forward and backward
        h = torch.cat([h[0], h[1]], dim=1)
        return self.fc(h)

model = TextRNN(vocab_size=100, embed_dim=32, hidden_dim=64, num_classes=2)
print(f"BiLSTM output: {model(x).shape}")

## 4. Transformer-alapú osztályozás (BERT-style)

A modern megközelítés: előtanított Transformer encoder + [CLS] token klasszifikáció.

### BERT klasszifikáció architektúra

```
[CLS] Ez egy pozitív vélemény [SEP]
  ↓
Transformer Encoder (12 réteg)
  ↓
[CLS] reprezentáció (768 dim)
  ↓
Linear(768 → num_classes)
  ↓
Softmax → Osztály valószínűségek
```

### Miért [CLS] token?

- A **[CLS]** (classification) token speciális: a mondat elejére kerül
- A self-attention miatt "látja" az összes tokent
- A Transformer output-ja ezen a pozíción a **mondat reprezentációja**

### Fine-tuning stratégia

1. **Pre-trained BERT** betöltése (pl. bert-base-uncased)
2. **Klasszifikációs fej** hozzáadása (Linear layer)
3. **Fine-tuning**: az egész modellt tanítjuk a specifikus feladatra

In [ ]:
class SimpleTextTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        encoder_layer = nn.TransformerEncoderLayer(d_model, n_heads, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        B = x.shape[0]
        emb = self.embedding(x)
        # [CLS] token hozzáadása
        cls = self.cls_token.expand(B, -1, -1)
        emb = torch.cat([cls, emb], dim=1)

        out = self.encoder(emb)
        return self.fc(out[:, 0])  # [CLS] token kimenet

model = SimpleTextTransformer(vocab_size=100, d_model=64, n_heads=4, num_classes=2)
print(f"Transformer output: {model(x).shape}")

## 5. Teljes pipeline gyakorlat

Készítsünk egy egyszerű sentiment analysis modellt!

In [ ]:
# Adatok előkészítése
train_texts = [
    "Kiváló termék, nagyon elégedett vagyok",
    "Szörnyű minőség, visszaküldöm",
    "Ajánlom mindenkinek, remek választás",
    "Csalódás, nem működik rendesen",
    "Fantasztikus, megérte az árát",
    "Rossz, ne vedd meg",
    "Tökéletes, pont amit kerestem",
    "Borzalmas élmény, soha többet",
]
train_labels = [1, 0, 1, 0, 1, 0, 1, 0]  # 1=pozitív, 0=negatív

# Szótár építése
def build_vocab_from_texts(texts, min_freq=1):
    counter = Counter()
    for text in texts:
        counter.update(tokenize(text))

    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab

vocab = build_vocab_from_texts(train_texts)
print(f"Vocab size: {len(vocab)}")

In [ ]:
# Szöveg → tensor konverzió
def text_to_tensor(text, vocab, max_len=20):
    tokens = tokenize(text)
    indices = [vocab.get(t, vocab['<UNK>']) for t in tokens]

    # Padding vagy truncate
    if len(indices) < max_len:
        indices += [vocab['<PAD>']] * (max_len - len(indices))
    else:
        indices = indices[:max_len]

    return torch.tensor(indices)

# Dataset készítése
X_train = torch.stack([text_to_tensor(t, vocab) for t in train_texts])
y_train = torch.tensor(train_labels)

print(f"X_train shape: {X_train.shape}")
print(f"Sample encoded: {X_train[0][:10]}")

In [ ]:
# Modell tanítása
model = TextCNN(vocab_size=len(vocab), embed_dim=32, num_classes=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

# Training loop
losses = []
for epoch in range(50):
    optimizer.zero_grad()
    output = model(X_train)
    loss = criterion(output, y_train)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    if (epoch + 1) % 10 == 0:
        preds = output.argmax(dim=1)
        acc = (preds == y_train).float().mean()
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}, Acc: {acc:.2%}")

plt.plot(losses)
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('TextCNN Training')
plt.show()

In [ ]:
# Predikció új szövegekre
test_texts = [
    "Nagyon jó, örülök neki",
    "Nem tetszik, rossz választás volt",
]

model.eval()
with torch.no_grad():
    for text in test_texts:
        x = text_to_tensor(text, vocab).unsqueeze(0)
        output = model(x)
        prob = F.softmax(output, dim=1)
        pred = output.argmax(dim=1).item()
        label = "POZITÍV" if pred == 1 else "NEGATÍV"
        confidence = prob[0, pred].item()
        print(f'"{text}" → {label} ({confidence:.1%})')

## Összefoglalás

### Módszerek összehasonlítása

| Módszer | Előny | Hátrány | Mikor használd? |
|---------|-------|---------|-----------------|
| **BoW/TF-IDF** | Egyszerű, gyors, interpretálható | Nincs sorrend, sparse | Baseline, kis adat |
| **TextCNN** | Lokális n-gram minták | Fix receptív mező | Rövid szövegek |
| **BiLSTM** | Szekvenciális kontextus | Lassú, gradiens problémák | Közepes hossz |
| **Transformer** | Globális kontextus | Számításigényes | Legjobb minőség |

### Gyakorlati tippek

1. **Baseline**: Mindig kezdj egyszerű modellel (TF-IDF + Logistic Regression)
2. **Pre-trained**: Használj előtanított embeddings-eket (Word2Vec, FastText)
3. **Fine-tuning**: BERT/RoBERTa fine-tuning a state-of-the-art
4. **Class imbalance**: Használj weighted loss-t vagy oversampling-et
5. **Evaluation**: Accuracy mellett nézd a precision, recall, F1-et

### HuggingFace példa (éles használat)

```python
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")
result = classifier("Ez egy kiváló termék!")
print(result)  # [{'label': '5 stars', 'score': 0.89}]
```

### Következő lépések

A következő notebookban a **pre-trained language models** (BERT, GPT) működését és használatát nézzük részletesen!